In [ ]:
import nltk
import re
nltk.download('punkt')
nltk.download('punkt_tab')
text ="""Palestine is a country in West Asia.
It comprises the West Bank and Gaza Strip.
میری جنت، میری ماں
"""
from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(text)
print(sentences)
from nltk.tokenize import word_tokenize
words = word_tokenize(text)
print(words)
text2 = re.sub(r"[^a-zA-Z0-9]"," ",text)
print(text)
print(text2)
tokens1 = text.split()
print(tokens1)
print(len(tokens1))
tokens2 = re.findall(r"\w+", text)
print(tokens2)
print(len(tokens2))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['Palestine is a country in West Asia.', 'It comprises the West Bank and Gaza Strip.', 'میری جنت، میری ماں']
['Palestine', 'is', 'a', 'country', 'in', 'West', 'Asia', '.', 'It', 'comprises', 'the', 'West', 'Bank', 'and', 'Gaza', 'Strip', '.', 'میری', 'جنت،', 'میری', 'ماں']
Palestine is a country in West Asia.
It comprises the West Bank and Gaza Strip.
میری جنت، میری ماں

Palestine is a country in West Asia  It comprises the West Bank and Gaza Strip                     
['Palestine', 'is', 'a', 'country', 'in', 'West', 'Asia.', 'It', 'comprises', 'the', 'West', 'Bank', 'and', 'Gaza', 'Strip.', 'میری', 'جنت،', 'میری', 'ماں']
19
['Palestine', 'is', 'a', 'country', 'in', 'West', 'Asia', 'It', 'comprises', 'the', 'West', 'Bank', 'and', 'Gaza', 'Strip', 'میری', 'جنت', 'میری', 'ماں']
19


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def chatbot_app():

    print("Welcome to AI World")
    print("--- Type 'exit' to exit ---")

    FAQ = [
        ["What is Artificial Intelligence?",
         "AI is a branch of computer science that enables machines to perform tasks requiring human intelligence."],

        ["What is Machine Learning?",
         "Machine Learning allows computers to learn from data."],

        ["What is Deep Learning?",
         "Deep Learning uses neural networks to learn complex patterns."],

        ["What is a chatbot?",
         "A chatbot is a program that communicates with users."],

        ["What is a neural network?",
         "It is a computing model inspired by the human brain."],

        ["What is a dataset?",
         "A dataset is a collection of information used for analysis or training."],

        ["What is training data?",
         "Training data is used to teach an AI model."],

        ["What is Computer Vision?",
         "It enables computers to understand images and videos."]
    ]

    questions = [item[0] for item in FAQ]

    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(questions)

    while True:

        user_input = input("Question: ")

        if user_input.lower() == "exit":
            print("Chatbot: Goodbye!")
            break

        user_question = vectorizer.transform([user_input])

        score = cosine_similarity(user_question, tfidf)

        match = score.argmax()

        if score[0][match] >= 0.20:
            print("Chatbot:", FAQ[match][1])
        else:
            print("Chatbot: Sorry, I don't understand your question.")

chatbot_app()


Welcome to AI World
--- Type 'exit' to exit ---
Chatbot: It is a computing model inspired by the human brain.
Chatbot: Sorry, I don't understand your question.


In [ ]:

import re
from lab6_search import Retriever, show_hits
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
def windows(text, size=40, overlap=10, offset=0):

    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Invalid chunk size or overlap")

    spans = word_spans(text)
    result = []

    for i in range(0, len(spans), size - overlap):

        j = min(i + size, len(spans))

        result.append((
            offset + spans[i][0],
            offset + spans[j - 1][1]
        ))

        if j == len(spans):
            break

    return result
def sentence_chunks(text, size=40):

    if size <= 0:
        raise ValueError("Size must be positive")

    cuts = [0] + [
        m.end() for m in re.finditer(r"[.!?]\s+", text)
    ] + [len(text)]

    out = []
    pending = None
    count = 0

    for a, b in zip(cuts, cuts[1:]):

        segment = text[a:b]
        n = len(segment.split())

        if not n:
            continue

        if n > size:

            if pending is not None:
                out.append(pending)

            out.extend(windows(segment, size, 0, a))

            pending = None
            count = 0

        elif pending is not None and count + n <= size:

            pending = (pending[0], b)
            count += n

        else:

            if pending is not None:
                out.append(pending)

            pending = (a, b)
            count = n

    if pending is not None:
        out.append(pending)

    return out
DOCS = {

    "D1": """
The university learning portal helps students manage their academic work throughout the semester. Students can read announcements, download lecture notes, submit assignments, and check their marks. Every student should use a personal account and keep the password private.

Rule: Students must submit each assignment before the deadline shown on the portal.
Exception: A late submission may be accepted when the student has an approved medical certificate or written permission from the course instructor.

Students should contact the instructor as soon as a problem occurs. The portal records the submission time, so students should not wait until the final minute. A student who has a technical problem should take a screenshot and report the issue to the support office with the course name and assignment title. Teachers review submitted work and may provide feedback through the same portal. Students are responsible for checking that the correct file was uploaded and that the file opens properly. The system may become slow during busy periods, especially near deadlines, so early submission is recommended. The university also provides a help desk for account and upload problems. Students should read course instructions carefully before submitting any work. If a student cannot access the portal, the student should contact the support team, explain the problem clearly, and provide enough information so that the staff can investigate the issue and help restore access without delaying academic activities.
""",

    "D2": """
The campus library provides books, computers, study rooms, and online learning resources for students. Students can borrow books by presenting their university identification card.

Rule: Each student may borrow up to three books for a period of fourteen days.
Exception: A student working on an approved research project may request one additional book from the librarian.

Borrowed books must be returned on or before the due date. The library sends reminders to students when a return date is approaching. A student who loses a book should inform the library staff and follow the replacement procedure. Quiet study areas are available on the first floor, while group discussion rooms can be booked at the front desk. Food and drinks are not allowed near computers or library books. Students should keep their phones silent and avoid disturbing other readers. The library website lists available books and provides access to digital journals. If a book is not available, students can ask staff about other editions or interlibrary requests. During examination weeks, the library may extend its opening hours. Updated timings are displayed at the entrance and on the university website. Students should check these notices before visiting. Library staff can also help students find books related to their subjects and research projects.
""",
import csv
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from task2_chunks import make_chunks, DOCS
DOCS["D4"] = """
The university attendance policy explains the minimum class attendance required
for students during an academic semester. Students should attend lectures,
practical sessions, and other scheduled academic activities. Attendance records
are maintained by the department and checked by teachers regularly.

Rule: Students must maintain at least 75% attendance to be eligible for
the final examination.

Exception: Students with approved medical leave may request special consideration
from the department, but approval is not automatic.

Students should check their attendance records throughout the semester.
If attendance is incorrect, they should contact the relevant teacher.
The department recommends that students attend classes regularly and avoid
unnecessary absences. Students who miss classes should collect lecture notes
and complete any missed activities. Teachers may inform students when their
attendance becomes low. The university expects students to understand the
attendance policy before the semester begins. Academic advisors can provide
guidance about attendance problems and departmental procedures.
"""


DOCS["D5"] = """
The academic support office provides guidance about examination eligibility
and student attendance. Attendance is important because it helps students
participate in lectures and practical activities. Department staff maintain
attendance records and share information with students when required.

Rule: Students must maintain at least 80% attendance to qualify for
the final examination according to the academic support office policy.

Exception: Students with an officially approved medical leave may apply
for special consideration through the department.

Students should review their attendance records and report any mistakes.
Teachers may advise students who have missed several classes. Students
should contact their academic advisor when they need information about
examination requirements. The support office also explains university
procedures and academic responsibilities. Students are encouraged to
attend all scheduled classes and complete their academic work on time.
This document represents a separate policy statement and may conflict
with attendance information published by another university department.
"""

items = [
    {
        "query": "What happens if I submit my assignment late?",
        "split": "dev",
        "doc": "D1",
        "evidence": "A late submission may be accepted when the student has an approved medical certificate or written permission from the course instructor."
    },

    {
        "query": "How many books can I borrow at once?",
        "split": "dev",
        "doc": "D2",
        "evidence": "Each student may borrow up to three books for a period of fourteen days."
    },

    {
        "query": "Can I use another computer if mine is not working?",
        "split": "dev",
        "doc": "D3",
        "evidence": "If the assigned computer is not working, the student may use another available computer after receiving permission from the lab assistant."
    },

    {
        "query": "What is the minimum attendance required for exams?",
        "split": "dev",
        "doc": "D4",
        "evidence": "Students must maintain at least 75% attendance to be eligible for"
    },

    {
        "query": "What attendance percentage does the academic support office require?",
        "split": "test",
        "doc": "D5",
        "evidence": "Students must maintain at least 80% attendance to qualify for"
    },

    {
        "query": "What should I do when my assignment upload fails?",
        "split": "test",
        "doc": "D1",
        "evidence": "A student who has a technical problem should take a screenshot and report the issue to the support office with the course name and assignment title."
    },

    {
        "query": "Can research students borrow an extra library book?",
        "split": "test",
        "doc": "D2",
        "evidence": "A student working on an approved research project may request one additional book from the librarian."
    },

    {
        "query": "What should students do before leaving the computer laboratory?",
        "split": "test",
        "doc": "D3",
        "evidence": "Students must save their work in their assigned folder before leaving the laboratory."
    },

    {
        "query": "What is the university tuition refund deadline?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    },

    {
        "query": "Does the library provide free meals to students?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    }
]

def evaluate(scores, chunks, questions):

    hits = []
    reciprocal = []

    for row, q in zip(scores, questions):

        if q["doc"] is None:
            continue

        top = np.argsort(-row, kind="stable")[:3]

        good = [
            rank
            for rank, i in enumerate(top, 1)
            if chunks[i]["doc"] == q["doc"]
            and q["evidence"] in chunks[i]["text"]
        ]

        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)

    if not hits:
        return None, None

    return float(np.mean(hits)), float(np.mean(reciprocal))
def calibrate_threshold(scores, questions):

    dev_scores = []
    labels = []

    for row, q in zip(scores, questions):

        if q["split"] != "dev":
            continue

        dev_scores.append(float(np.max(row)))
        labels.append(q["doc"] is not None)

    candidates = sorted(set([0.0, 1.0] + dev_scores))

    best_threshold = 0.0
    best_accuracy = -1

    for threshold in candidates:

        predictions = [
            score >= threshold for score in dev_scores
        ]

        positive = [
            i for i, label in enumerate(labels) if label
        ]

        negative = [
            i for i, label in enumerate(labels) if not label
        ]

        if not positive or not negative:
            continue

        tpr = sum(predictions[i] for i in positive) / len(positive)

        tnr = sum(not predictions[i] for i in negative) / len(negative)

        balanced_accuracy = (tpr + tnr) / 2

        if balanced_accuracy > best_accuracy:
            best_accuracy = balanced_accuracy
            best_threshold = threshold

    return best_threshold, best_accuracy


def main():
    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "dev"]) == 4

    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "test"]) == 4

    assert len([q for q in items if q["doc"] is None]) == 2

    for q in items:
        if q["doc"] is not None:
            assert q["evidence"] in DOCS[q["doc"]]

    print("Total documents:", len(DOCS))
    print("Total queries:", len(items))
    start = time.perf_counter()

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )

    load_time = time.perf_counter() - start

    print("\nModel load seconds:", load_time)
    print("Model maximum sequence length:", model.max_seq_length)

    query_texts = [q["query"] for q in items]

    logs = []
    for strategy in ["0", "10", "sentence"]:

        chunks = make_chunks(DOCS, strategy)

        texts = [c["text"] for c in chunks]

        print("\n================================")
        print("STRATEGY:", strategy)
        print("Total chunks:", len(chunks))
        lengths = [
            len(model.tokenizer(
                text, truncation=False
            )["input_ids"])
            for text in texts + query_texts
        ]

        print("Maximum token length:", max(lengths))
        print("Minimum token length:", min(lengths))
        print("Model token limit:", model.max_seq_length)

        if max(lengths) > model.max_seq_length:
            print("WARNING: Some text may be truncated by the model.")
        for method in ["tfidf", "embedding"]:

            start = time.perf_counter()

            if method == "tfidf":

                vectorizer = TfidfVectorizer()

                C = vectorizer.fit_transform(texts)
                Q = vectorizer.transform(query_texts)

            else:

                C = model.encode(
                    texts,
                    normalize_embeddings=True
                )

                Q = model.encode(
                    query_texts,
                    normalize_embeddings=True
                )

            encode_seconds = time.perf_counter() - start

            # Measure search time separately
            start = time.perf_counter()

            if method == "tfidf":
                scores = (Q @ C.T).toarray()
            else:
                scores = Q @ C.T

            search_seconds = time.perf_counter() - start
            threshold, dev_accuracy = calibrate_threshold(
                scores, items
            )

            print("\nBackend:", method)

            print("Encoding seconds:", encode_seconds)
            print("Search seconds:", search_seconds)

            print("Development threshold:", threshold)
            print("Development balanced accuracy:", dev_accuracy)
            for split in ["dev", "test"]:

                indices = [
                    i for i, q in enumerate(items)
                    if q["split"] == split
                    and q["doc"] is not None
                ]

                result = evaluate(
                    scores[indices],
                    chunks,
                    [items[i] for i in indices]
                )

                print(
                    strategy,
                    method,
                    split,
                    "Hit@3 / MRR@3:",
                    result
                )
                if split == "test":

                    accepted = sum(
                        float(np.max(scores[i])) >= threshold
                        for i in indices
                    )

                    print(
                        "Held-out accepted queries:",
                        accepted,
                        "out of",
                        len(indices)
                    )
            for row, q in zip(scores, items):

                print("\nQuery:", q["query"])
                print("Split:", q["split"])

                for rank, i in enumerate(
                    np.argsort(-row, kind="stable")[:3], 1
                ):

                    c = chunks[i]

                    record = {
                        "strategy": strategy,
                        "method": method,
                        "query": q["query"],
                        "split": q["split"],
                        "rank": rank,
                        "score": float(row[i]),
                        "chunk_id": c["id"],
                        "doc": c["doc"],
                        "start": c["start"],
                        "end": c["end"],
                        "text": c["text"],
                        "answerable": q["doc"] is not None,
                        "threshold": threshold
                    }

                    logs.append(record)

                    print("\nRank:", rank)
                    print("Score:", float(row[i]))
                    print("Chunk ID:", c["id"])
                    print("Document:", c["doc"])
                    print("Offsets:", c["start"], c["end"])
                    print("Original evidence:", c["text"])
                if q["doc"] is None:
                    print(
                        "Unanswerable query: candidates are not "
                        "automatically accepted as answers."
                    )

import csv
import time
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

from task2_chunks import make_chunks, DOCS


# Add two more documents with related but conflicting information

DOCS["D4"] = """
The university attendance policy explains the minimum class attendance required
for students during an academic semester. Students should attend lectures,
practical sessions, and other scheduled academic activities. Attendance records
are maintained by the department and checked by teachers regularly.

Rule: Students must maintain at least 75% attendance to be eligible for
the final examination.

Exception: Students with approved medical leave may request special consideration
from the department, but approval is not automatic.

Students should check their attendance records throughout the semester.
If attendance is incorrect, they should contact the relevant teacher.
The department recommends that students attend classes regularly and avoid
unnecessary absences. Students who miss classes should collect lecture notes
and complete any missed activities. Teachers may inform students when their
attendance becomes low. The university expects students to understand the
attendance policy before the semester begins. Academic advisors can provide
guidance about attendance problems and departmental procedures.
"""


DOCS["D5"] = """
The academic support office provides guidance about examination eligibility
and student attendance. Attendance is important because it helps students
participate in lectures and practical activities. Department staff maintain
attendance records and share information with students when required.

Rule: Students must maintain at least 80% attendance to qualify for
the final examination according to the academic support office policy.

Exception: Students with an officially approved medical leave may apply
for special consideration through the department.

Students should review their attendance records and report any mistakes.
Teachers may advise students who have missed several classes. Students
should contact their academic advisor when they need information about
examination requirements. The support office also explains university
procedures and academic responsibilities. Students are encouraged to
attend all scheduled classes and complete their academic work on time.
This document represents a separate policy statement and may conflict
with attendance information published by another university department.
"""


# Eight answerable queries and two unanswerable queries
# Four answerable queries for development
# Four answerable queries for final testing

items = [

    {
        "query": "What happens if I submit my assignment late?",
        "split": "dev",
        "doc": "D1",
        "evidence": "A late submission may be accepted when the student has an approved medical certificate or written permission from the course instructor."
    },

    {
        "query": "How many books can I borrow at once?",
        "split": "dev",
        "doc": "D2",
        "evidence": "Each student may borrow up to three books for a period of fourteen days."
    },

    {
        "query": "Can I use another computer if mine is not working?",
        "split": "dev",
        "doc": "D3",
        "evidence": "If the assigned computer is not working, the student may use another available computer after receiving permission from the lab assistant."
    },

    {
        "query": "What is the minimum attendance required for exams?",
        "split": "dev",
        "doc": "D4",
        "evidence": "Students must maintain at least 75% attendance to be eligible for"
    },

    {
        "query": "What attendance percentage does the academic support office require?",
        "split": "test",
        "doc": "D5",
        "evidence": "Students must maintain at least 80% attendance to qualify for"
    },

    {
        "query": "What should I do when my assignment upload fails?",
        "split": "test",
        "doc": "D1",
        "evidence": "A student who has a technical problem should take a screenshot and report the issue to the support office with the course name and assignment title."
    },

    {
        "query": "Can research students borrow an extra library book?",
        "split": "test",
        "doc": "D2",
        "evidence": "A student working on an approved research project may request one additional book from the librarian."
    },

    {
        "query": "What should students do before leaving the computer laboratory?",
        "split": "test",
        "doc": "D3",
        "evidence": "Students must save their work in their assigned folder before leaving the laboratory."
    },

    {
        "query": "What is the university tuition refund deadline?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    },

    {
        "query": "Does the library provide free meals to students?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    }
]


# Calculate Hit@3 and MRR@3
def evaluate(scores, chunks, questions):

    hits = []
    reciprocal = []

    for row, q in zip(scores, questions):

        if q["doc"] is None:
            continue

        top = np.argsort(-row, kind="stable")[:3]

        good = [
            rank
            for rank, i in enumerate(top, 1)
            if chunks[i]["doc"] == q["doc"]
            and q["evidence"] in chunks[i]["text"]
        ]

        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)

    if not hits:
        return None, None

    return float(np.mean(hits)), float(np.mean(reciprocal))


# Calibrate rejection threshold using development queries only
def calibrate_threshold(scores, questions):

    dev_scores = []
    labels = []

    for row, q in zip(scores, questions):

        if q["split"] != "dev":
            continue

        dev_scores.append(float(np.max(row)))
        labels.append(q["doc"] is not None)

    candidates = sorted(set([0.0, 1.0] + dev_scores))

    best_threshold = 0.0
    best_accuracy = -1

    for threshold in candidates:

        predictions = [
            score >= threshold for score in dev_scores
        ]

        positive = [
            i for i, label in enumerate(labels) if label
        ]

        negative = [
            i for i, label in enumerate(labels) if not label
        ]

        if not positive or not negative:
            continue

        tpr = sum(predictions[i] for i in positive) / len(positive)

        tnr = sum(not predictions[i] for i in negative) / len(negative)

        balanced_accuracy = (tpr + tnr) / 2

        if balanced_accuracy > best_accuracy:
            best_accuracy = balanced_accuracy
            best_threshold = threshold

    return best_threshold, best_accuracy


def main():

    # Check query labels
    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "dev"]) == 4

    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "test"]) == 4

    assert len([q for q in items if q["doc"] is None]) == 2

    for q in items:
        if q["doc"] is not None:
            assert q["evidence"] in DOCS[q["doc"]]

    print("Total documents:", len(DOCS))
    print("Total queries:", len(items))

    # Load real sentence embedding model
    start = time.perf_counter()

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )

    load_time = time.perf_counter() - start

    print("\nModel load seconds:", load_time)
    print("Model maximum sequence length:", model.max_seq_length)

    query_texts = [q["query"] for q in items]

    logs = []

    # Compare three chunking strategies
    for strategy in ["0", "10", "sentence"]:

        chunks = make_chunks(DOCS, strategy)

        texts = [c["text"] for c in chunks]

        print("\n================================")
        print("STRATEGY:", strategy)
        print("Total chunks:", len(chunks))

        # Audit model token lengths
        lengths = [
            len(model.tokenizer(
                text, truncation=False
            )["input_ids"])
            for text in texts + query_texts
        ]

        print("Maximum token length:", max(lengths))
        print("Minimum token length:", min(lengths))
        print("Model token limit:", model.max_seq_length)

        if max(lengths) > model.max_seq_length:
            print("WARNING: Some text may be truncated by the model.")

        # Compare TF-IDF and actual embeddings
        for method in ["tfidf", "embedding"]:

            start = time.perf_counter()

            if method == "tfidf":

                vectorizer = TfidfVectorizer()

                C = vectorizer.fit_transform(texts)
                Q = vectorizer.transform(query_texts)

            else:

                C = model.encode(
                    texts,
                    normalize_embeddings=True
                )

                Q = model.encode(
                    query_texts,
                    normalize_embeddings=True
                )

            encode_seconds = time.perf_counter() - start

            # Measure search time separately
            start = time.perf_counter()

            if method == "tfidf":
                scores = (Q @ C.T).toarray()
            else:
                scores = Q @ C.T

            search_seconds = time.perf_counter() - start

            # Development threshold calibration
            threshold, dev_accuracy = calibrate_threshold(
                scores, items
            )

            print("\nBackend:", method)

            print("Encoding seconds:", encode_seconds)
            print("Search seconds:", search_seconds)

            print("Development threshold:", threshold)
            print("Development balanced accuracy:", dev_accuracy)

            # Evaluate development and final test separately
            for split in ["dev", "test"]:

                indices = [
                    i for i, q in enumerate(items)
                    if q["split"] == split
                    and q["doc"] is not None
                ]

                result = evaluate(
                    scores[indices],
                    chunks,
                    [items[i] for i in indices]
                )

                print(
                    strategy,
                    method,
                    split,
                    "Hit@3 / MRR@3:",
                    result
                )

                # Apply the development threshold without retuning
                if split == "test":

                    accepted = sum(
                        float(np.max(scores[i])) >= threshold
                        for i in indices
                    )

                    print(
                        "Held-out accepted queries:",
                        accepted,
                        "out of",
                        len(indices)
                    )

            # Display top-three evidence and create CSV records
            for row, q in zip(scores, items):

                print("\nQuery:", q["query"])
                print("Split:", q["split"])

                for rank, i in enumerate(
                    np.argsort(-row, kind="stable")[:3], 1
                ):

                    c = chunks[i]

                    record = {
                        "strategy": strategy,
                        "method": method,
                        "query": q["query"],
                        "split": q["split"],
                        "rank": rank,
                        "score": float(row[i]),
                        "chunk_id": c["id"],
                        "doc": c["doc"],
                        "start": c["start"],
                        "end": c["end"],
                        "text": c["text"],
                        "answerable": q["doc"] is not None,
                        "threshold": threshold
                    }

                    logs.append(record)

                    print("\nRank:", rank)
                    print("Score:", float(row[i]))
                    print("Chunk ID:", c["id"])
                    print("Document:", c["doc"])
                    print("Offsets:", c["start"], c["end"])
                    print("Original evidence:", c["text"])

                # Unanswerable queries still return candidates
                if q["doc"] is None:
                    print(
                        "Unanswerable query: candidates are not "
                        "automatically accepted as answers."
                    )

    # Export all retrieval results
    with open(
        "retrieval_log.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=logs[0].keys()
        )

        writer.writeheader()
        writer.writerows(logs)

    print("\nCSV file saved: retrieval_log.csv")


if __name__ == "__main__":
    main()

    with open(
        "retrieval_log.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=logs[0].keys()
        )

        writer.writeheader()
        writer.writerows(logs)

    print("\nCSV file saved: retrieval_log.csv")


if __name__ == "__main__":
    main()


import csv
import time
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

from task2_chunks import make_chunks, DOCS


# Add two more documents with related but conflicting information

DOCS["D4"] = """
The university attendance policy explains the minimum class attendance required
for students during an academic semester. Students should attend lectures,
practical sessions, and other scheduled academic activities. Attendance records
are maintained by the department and checked by teachers regularly.

Rule: Students must maintain at least 75% attendance to be eligible for
the final examination.

Exception: Students with approved medical leave may request special consideration
from the department, but approval is not automatic.

Students should check their attendance records throughout the semester.
If attendance is incorrect, they should contact the relevant teacher.
The department recommends that students attend classes regularly and avoid
unnecessary absences. Students who miss classes should collect lecture notes
and complete any missed activities. Teachers may inform students when their
attendance becomes low. The university expects students to understand the
attendance policy before the semester begins. Academic advisors can provide
guidance about attendance problems and departmental procedures.
"""


DOCS["D5"] = """
The academic support office provides guidance about examination eligibility
and student attendance. Attendance is important because it helps students
participate in lectures and practical activities. Department staff maintain
attendance records and share information with students when required.

Rule: Students must maintain at least 80% attendance to qualify for
the final examination according to the academic support office policy.

Exception: Students with an officially approved medical leave may apply
for special consideration through the department.

Students should review their attendance records and report any mistakes.
Teachers may advise students who have missed several classes. Students
should contact their academic advisor when they need information about
examination requirements. The support office also explains university
procedures and academic responsibilities. Students are encouraged to
attend all scheduled classes and complete their academic work on time.
This document represents a separate policy statement and may conflict
with attendance information published by another university department.
"""


# Eight answerable queries and two unanswerable queries
# Four answerable queries for development
# Four answerable queries for final testing

items = [

    {
        "query": "What happens if I submit my assignment late?",
        "split": "dev",
        "doc": "D1",
        "evidence": "A late submission may be accepted when the student has an approved medical certificate or written permission from the course instructor."
    },

    {
        "query": "How many books can I borrow at once?",
        "split": "dev",
        "doc": "D2",
        "evidence": "Each student may borrow up to three books for a period of fourteen days."
    },

    {
        "query": "Can I use another computer if mine is not working?",
        "split": "dev",
        "doc": "D3",
        "evidence": "If the assigned computer is not working, the student may use another available computer after receiving permission from the lab assistant."
    },

    {
        "query": "What is the minimum attendance required for exams?",
        "split": "dev",
        "doc": "D4",
        "evidence": "Students must maintain at least 75% attendance to be eligible for"
    },

    {
        "query": "What attendance percentage does the academic support office require?",
        "split": "test",
        "doc": "D5",
        "evidence": "Students must maintain at least 80% attendance to qualify for"
    },

    {
        "query": "What should I do when my assignment upload fails?",
        "split": "test",
        "doc": "D1",
        "evidence": "A student who has a technical problem should take a screenshot and report the issue to the support office with the course name and assignment title."
    },

    {
        "query": "Can research students borrow an extra library book?",
        "split": "test",
        "doc": "D2",
        "evidence": "A student working on an approved research project may request one additional book from the librarian."
    },

    {
        "query": "What should students do before leaving the computer laboratory?",
        "split": "test",
        "doc": "D3",
        "evidence": "Students must save their work in their assigned folder before leaving the laboratory."
    },

    {
        "query": "What is the university tuition refund deadline?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    },

    {
        "query": "Does the library provide free meals to students?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    }
]


# Calculate Hit@3 and MRR@3
def evaluate(scores, chunks, questions):

    hits = []
    reciprocal = []

    for row, q in zip(scores, questions):

        if q["doc"] is None:
            continue

        top = np.argsort(-row, kind="stable")[:3]

        good = [
            rank
            for rank, i in enumerate(top, 1)
            if chunks[i]["doc"] == q["doc"]
            and q["evidence"] in chunks[i]["text"]
        ]

        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)

    if not hits:
        return None, None

    return float(np.mean(hits)), float(np.mean(reciprocal))


# Calibrate rejection threshold using development queries only
def calibrate_threshold(scores, questions):

    dev_scores = []
    labels = []

    for row, q in zip(scores, questions):

        if q["split"] != "dev":
            continue

        dev_scores.append(float(np.max(row)))
        labels.append(q["doc"] is not None)

    candidates = sorted(set([0.0, 1.0] + dev_scores))

    best_threshold = 0.0
    best_accuracy = -1

    for threshold in candidates:

        predictions = [
            score >= threshold for score in dev_scores
        ]

        positive = [
            i for i, label in enumerate(labels) if label
        ]

        negative = [
            i for i, label in enumerate(labels) if not label
        ]

        if not positive or not negative:
            continue

        tpr = sum(predictions[i] for i in positive) / len(positive)

        tnr = sum(not predictions[i] for i in negative) / len(negative)

        balanced_accuracy = (tpr + tnr) / 2

        if balanced_accuracy > best_accuracy:
            best_accuracy = balanced_accuracy
            best_threshold = threshold

    return best_threshold, best_accuracy


def main():

    # Check query labels
    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "dev"]) == 4

    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "test"]) == 4

    assert len([q for q in items if q["doc"] is None]) == 2

    for q in items:
        if q["doc"] is not None:
            assert q["evidence"] in DOCS[q["doc"]]

    print("Total documents:", len(DOCS))
    print("Total queries:", len(items))

    # Load real sentence embedding model
    start = time.perf_counter()

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )

    load_time = time.perf_counter() - start

    print("\nModel load seconds:", load_time)
    print("Model maximum sequence length:", model.max_seq_length)

    query_texts = [q["query"] for q in items]

    logs = []

    # Compare three chunking strategies
    for strategy in ["0", "10", "sentence"]:

        chunks = make_chunks(DOCS, strategy)

        texts = [c["text"] for c in chunks]

        print("\n================================")
        print("STRATEGY:", strategy)
        print("Total chunks:", len(chunks))

        # Audit model token lengths
        lengths = [
            len(model.tokenizer(
                text, truncation=False
            )["input_ids"])
            for text in texts + query_texts
        ]

        print("Maximum token length:", max(lengths))
        print("Minimum token length:", min(lengths))
        print("Model token limit:", model.max_seq_length)

        if max(lengths) > model.max_seq_length:
            print("WARNING: Some text may be truncated by the model.")

        # Compare TF-IDF and actual embeddings
        for method in ["tfidf", "embedding"]:

            start = time.perf_counter()

            if method == "tfidf":

                vectorizer = TfidfVectorizer()

                C = vectorizer.fit_transform(texts)
                Q = vectorizer.transform(query_texts)

            else:

                C = model.encode(
                    texts,
                    normalize_embeddings=True
                )

                Q = model.encode(
                    query_texts,
                    normalize_embeddings=True
                )

            encode_seconds = time.perf_counter() - start

            # Measure search time separately
            start = time.perf_counter()

            if method == "tfidf":
                scores = (Q @ C.T).toarray()
            else:
                scores = Q @ C.T

            search_seconds = time.perf_counter() - start

            # Development threshold calibration
            threshold, dev_accuracy = calibrate_threshold(
                scores, items
            )

            print("\nBackend:", method)

            print("Encoding seconds:", encode_seconds)
            print("Search seconds:", search_seconds)

            print("Development threshold:", threshold)
            print("Development balanced accuracy:", dev_accuracy)

            # Evaluate development and final test separately
            for split in ["dev", "test"]:

                indices = [
                    i for i, q in enumerate(items)
                    if q["split"] == split
                    and q["doc"] is not None
                ]

                result = evaluate(
                    scores[indices],
                    chunks,
                    [items[i] for i in indices]
                )

                print(
                    strategy,
                    method,
                    split,
                    "Hit@3 / MRR@3:",
                    result
                )

                # Apply the development threshold without retuning
                if split == "test":

                    accepted = sum(
                        float(np.max(scores[i])) >= threshold
                        for i in indices
                    )

                    print(
                        "Held-out accepted queries:",
                        accepted,
                        "out of",
                        len(indices)
                    )

            # Display top-three evidence and create CSV records
            for row, q in zip(scores, items):

                print("\nQuery:", q["query"])
                print("Split:", q["split"])

                for rank, i in enumerate(
                    np.argsort(-row, kind="stable")[:3], 1
                ):

                    c = chunks[i]

                    record = {
                        "strategy": strategy,
                        "method": method,
                        "query": q["query"],
                        "split": q["split"],
                        "rank": rank,
                        "score": float(row[i]),
                        "chunk_id": c["id"],
                        "doc": c["doc"],
                        "start": c["start"],
                        "end": c["end"],
                        "text": c["text"],
                        "answerable": q["doc"] is not None,
                        "threshold": threshold
                    }

                    logs.append(record)

                    print("\nRank:", rank)
                    print("Score:", float(row[i]))
                    print("Chunk ID:", c["id"])
                    print("Document:", c["doc"])
                    print("Offsets:", c["start"], c["end"])
                    print("Original evidence:", c["text"])

                # Unanswerable queries still return candidates
                if q["doc"] is None:
                    print(
                        "Unanswerable query: candidates are not "
                        "automatically accepted as answers."
                    )

    # Export all retrieval results
    with open(
        "retrieval_log.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=logs[0].keys()
        )

        writer.writeheader()
        writer.writerows(logs)

    print("\nCSV file saved: retrieval_log.csv")


if __name__ == "__main__":
    main()

}

LABELS = {
    D1: {
        "rule": "Students must submit each assignment before the deadline shown on the portal.",
        "exception": "A late submission may be accepted when the student has an approved medical certificate or written permission from the course instructor."
    },

    D2: {
        "rule": "Each student may borrow up to three books for a period of fourteen days.",
        "exception": "A student working on an approved research project may request one additional book from the librarian."
    },

    D3: {
        "rule": "Students must save their work in their assigned folder before leaving the laboratory.",
        "exception": "If the assigned computer is not working, the student may use another available computer after receiving permission from the lab assistant."
    }
}

 Create chunks with IDs and source offsets
def make_chunks(docs, strategy):

    result = []

    for doc_id, text in docs.items():

        if strategy == "sentence":
            spans = sentence_chunks(text, 40)

        else:
            spans = windows(text, 40, int(strategy))

        for i, (a, b) in enumerate(spans):

            result.append({
                "id": f"{doc_id}:{i}",
                "doc": doc_id,
                "start": a,
                "end": b,
                "text": text[a:b]
            })

    return result
def check_labels(chunks):

    for doc_id, labels in LABELS.items():

        rule_found = False
        exception_found = False
        together = False

        for chunk in chunks:

            if chunk["doc"] == doc_id:

                has_rule = labels["rule"] in chunk["text"]
                has_exception = labels["exception"] in chunk["text"]

                if has_rule:
                    rule_found = True

                if has_exception:
                    exception_found = True

                if has_rule and has_exception:
                    together = True

        print("\nDocument:", doc_id)
        print("Rule survives:", rule_found)
        print("Exception survives:", exception_found)
        print("Both in one chunk:", together)
def compare_chunks(strategy, chunks):

    print("\nStrategy:", strategy)

    for doc_id, text in DOCS.items():

        selected = [
            c for c in chunks if c["doc"] == doc_id
        ]

        original_words = len(word_spans(text))

        sizes = [
            len(word_spans(c["text"])) for c in selected
        ]

        repeated = sum(sizes) - original_words

        print("\nDocument:", doc_id)
        print("Total chunks:", len(selected))
        print("Chunk sizes:", sizes)
        print("Repeated words:", repeated)

if __name__ == "__main__":


    for doc_id, text in DOCS.items():

        total = len(word_spans(text))

        print(doc_id, "Word count:", total)

        assert 150 <= total <= 250
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)

    except ValueError:
        print("\nInvalid overlap test passed")

    else:
        raise AssertionError("Invalid overlap was accepted")
    long = " ".join(["word"] * 95) + "."

    long_chunks = sentence_chunks(long)

    assert all(
        len(word_spans(long[a:b])) <= 40
        for a, b in long_chunks
    )

    print("Long sentence test passed")
    for strategy in ["0", "10", "sentence"]:

        chunks = make_chunks(DOCS, strategy)

        compare_chunks(strategy, chunks)

        check_labels(chunks)
        for c in chunks:

            assert c["text"] == DOCS[c["doc"]][
                c["start"]:c["end"]
            ]

        print("\nTotal chunks:", len(chunks))
    records = make_chunks(DOCS, "sentence")

    queries = [
        "How can I submit my assignment late?",
        "How many books can a student borrow?",
        "What should I do if my computer is not working?"
    ]

    for backend in ["tfidf", "embedding"]:

        engine = Retriever(records, backend)

        print("\nBackend:", backend)

        for query in queries:

            print("\nQuestion:", query)

            results = engine.search(query, k=2)

            show_hits(results)

    print("\nAll tests completed!")


In [ ]:

import csv
import time
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

from task2_chunks import make_chunks, DOCS

DOCS["D4"] = """
The university attendance policy explains the minimum class attendance required
for students during an academic semester. Students should attend lectures,
practical sessions, and other scheduled academic activities. Attendance records
are maintained by the department and checked by teachers regularly.

Rule: Students must maintain at least 75% attendance to be eligible for
the final examination.

Exception: Students with approved medical leave may request special consideration
from the department, but approval is not automatic.

Students should check their attendance records throughout the semester.
If attendance is incorrect, they should contact the relevant teacher.
The department recommends that students attend classes regularly and avoid
unnecessary absences. Students who miss classes should collect lecture notes
and complete any missed activities. Teachers may inform students when their
attendance becomes low. The university expects students to understand the
attendance policy before the semester begins. Academic advisors can provide
guidance about attendance problems and departmental procedures.
"""


DOCS["D5"] = """
The academic support office provides guidance about examination eligibility
and student attendance. Attendance is important because it helps students
participate in lectures and practical activities. Department staff maintain
attendance records and share information with students when required.

Rule: Students must maintain at least 80% attendance to qualify for
the final examination according to the academic support office policy.

Exception: Students with an officially approved medical leave may apply
for special consideration through the department.

Students should review their attendance records and report any mistakes.
Teachers may advise students who have missed several classes. Students
should contact their academic advisor when they need information about
examination requirements. The support office also explains university
procedures and academic responsibilities. Students are encouraged to
attend all scheduled classes and complete their academic work on time.
This document represents a separate policy statement and may conflict
with attendance information published by another university department.
"""


items = [

    {
        "query": "What happens if I submit my assignment late?",
        "split": "dev",
        "doc": "D1",
        "evidence": "A late submission may be accepted when the student has an approved medical certificate or written permission from the course instructor."
    },

    {
        "query": "How many books can I borrow at once?",
        "split": "dev",
        "doc": "D2",
        "evidence": "Each student may borrow up to three books for a period of fourteen days."
    },

    {
        "query": "Can I use another computer if mine is not working?",
        "split": "dev",
        "doc": "D3",
        "evidence": "If the assigned computer is not working, the student may use another available computer after receiving permission from the lab assistant."
    },

    {
        "query": "What is the minimum attendance required for exams?",
        "split": "dev",
        "doc": "D4",
        "evidence": "Students must maintain at least 75% attendance to be eligible for"
    },

    {
        "query": "What attendance percentage does the academic support office require?",
        "split": "test",
        "doc": "D5",
        "evidence": "Students must maintain at least 80% attendance to qualify for"
    },

    {
        "query": "What should I do when my assignment upload fails?",
        "split": "test",
        "doc": "D1",
        "evidence": "A student who has a technical problem should take a screenshot and report the issue to the support office with the course name and assignment title."
    },

    {
        "query": "Can research students borrow an extra library book?",
        "split": "test",
        "doc": "D2",
        "evidence": "A student working on an approved research project may request one additional book from the librarian."
    },

    {
        "query": "What should students do before leaving the computer laboratory?",
        "split": "test",
        "doc": "D3",
        "evidence": "Students must save their work in their assigned folder before leaving the laboratory."
    },

    {
        "query": "What is the university tuition refund deadline?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    },

    {
        "query": "Does the library provide free meals to students?",
        "split": "dev",
        "doc": None,
        "evidence": ""
    }
]


# Calculate Hit@3 and MRR@3
def evaluate(scores, chunks, questions):

    hits = []
    reciprocal = []

    for row, q in zip(scores, questions):

        if q["doc"] is None:
            continue

        top = np.argsort(-row, kind="stable")[:3]

        good = [
            rank
            for rank, i in enumerate(top, 1)
            if chunks[i]["doc"] == q["doc"]
            and q["evidence"] in chunks[i]["text"]
        ]

        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)

    if not hits:
        return None, None

    return float(np.mean(hits)), float(np.mean(reciprocal))


# Calibrate rejection threshold using development queries only
def calibrate_threshold(scores, questions):

    dev_scores = []
    labels = []

    for row, q in zip(scores, questions):

        if q["split"] != "dev":
            continue

        dev_scores.append(float(np.max(row)))
        labels.append(q["doc"] is not None)

    candidates = sorted(set([0.0, 1.0] + dev_scores))

    best_threshold = 0.0
    best_accuracy = -1

    for threshold in candidates:

        predictions = [
            score >= threshold for score in dev_scores
        ]

        positive = [
            i for i, label in enumerate(labels) if label
        ]

        negative = [
            i for i, label in enumerate(labels) if not label
        ]

        if not positive or not negative:
            continue

        tpr = sum(predictions[i] for i in positive) / len(positive)

        tnr = sum(not predictions[i] for i in negative) / len(negative)

        balanced_accuracy = (tpr + tnr) / 2

        if balanced_accuracy > best_accuracy:
            best_accuracy = balanced_accuracy
            best_threshold = threshold

    return best_threshold, best_accuracy


def main():

    # Check query labels
    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "dev"]) == 4

    assert len([q for q in items if q["doc"] is not None
                and q["split"] == "test"]) == 4

    assert len([q for q in items if q["doc"] is None]) == 2

    for q in items:
        if q["doc"] is not None:
            assert q["evidence"] in DOCS[q["doc"]]

    print("Total documents:", len(DOCS))
    print("Total queries:", len(items))

    # Load real sentence embedding model
    start = time.perf_counter()

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )

    load_time = time.perf_counter() - start

    print("\nModel load seconds:", load_time)
    print("Model maximum sequence length:", model.max_seq_length)

    query_texts = [q["query"] for q in items]

    logs = []

    # Compare three chunking strategies
    for strategy in ["0", "10", "sentence"]:

        chunks = make_chunks(DOCS, strategy)

        texts = [c["text"] for c in chunks]

        print("\n================================")
        print("STRATEGY:", strategy)
        print("Total chunks:", len(chunks))

        # Audit model token lengths
        lengths = [
            len(model.tokenizer(
                text, truncation=False
            )["input_ids"])
            for text in texts + query_texts
        ]

        print("Maximum token length:", max(lengths))
        print("Minimum token length:", min(lengths))
        print("Model token limit:", model.max_seq_length)

        if max(lengths) > model.max_seq_length:
            print("WARNING: Some text may be truncated by the model.")

        # Compare TF-IDF and actual embeddings
        for method in ["tfidf", "embedding"]:

            start = time.perf_counter()

            if method == "tfidf":

                vectorizer = TfidfVectorizer()

                C = vectorizer.fit_transform(texts)
                Q = vectorizer.transform(query_texts)

            else:

                C = model.encode(
                    texts,
                    normalize_embeddings=True
                )

                Q = model.encode(
                    query_texts,
                    normalize_embeddings=True
                )

            encode_seconds = time.perf_counter() - start

            # Measure search time separately
            start = time.perf_counter()

            if method == "tfidf":
                scores = (Q @ C.T).toarray()
            else:
                scores = Q @ C.T

            search_seconds = time.perf_counter() - start

            # Development threshold calibration
            threshold, dev_accuracy = calibrate_threshold(
                scores, items
            )

            print("\nBackend:", method)

            print("Encoding seconds:", encode_seconds)
            print("Search seconds:", search_seconds)

            print("Development threshold:", threshold)
            print("Development balanced accuracy:", dev_accuracy)

            # Evaluate development and final test separately
            for split in ["dev", "test"]:

                indices = [
                    i for i, q in enumerate(items)
                    if q["split"] == split
                    and q["doc"] is not None
                ]

                result = evaluate(
                    scores[indices],
                    chunks,
                    [items[i] for i in indices]
                )

                print(
                    strategy,
                    method,
                    split,
                    "Hit@3 / MRR@3:",
                    result
                )

                # Apply the development threshold without retuning
                if split == "test":

                    accepted = sum(
                        float(np.max(scores[i])) >= threshold
                        for i in indices
                    )

                    print(
                        "Held-out accepted queries:",
                        accepted,
                        "out of",
                        len(indices)
                    )

            # Display top-three evidence and create CSV records
            for row, q in zip(scores, items):

                print("\nQuery:", q["query"])
                print("Split:", q["split"])

                for rank, i in enumerate(
                    np.argsort(-row, kind="stable")[:3], 1
                ):

                    c = chunks[i]

                    record = {
                        "strategy": strategy,
                        "method": method,
                        "query": q["query"],
                        "split": q["split"],
                        "rank": rank,
                        "score": float(row[i]),
                        "chunk_id": c["id"],
                        "doc": c["doc"],
                        "start": c["start"],
                        "end": c["end"],
                        "text": c["text"],
                        "answerable": q["doc"] is not None,
                        "threshold": threshold
                    }

                    logs.append(record)

                    print("\nRank:", rank)
                    print("Score:", float(row[i]))
                    print("Chunk ID:", c["id"])
                    print("Document:", c["doc"])
                    print("Offsets:", c["start"], c["end"])
                    print("Original evidence:", c["text"])

                # Unanswerable queries still return candidates
                if q["doc"] is None:
                    print(
                        "Unanswerable query: candidates are not "
                        "automatically accepted as answers."
                    )

    # Export all retrieval results
    with open(
        "retrieval_log.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=logs[0].keys()
        )

        writer.writeheader()
        writer.writerows(logs)

    print("\nCSV file saved: retrieval_log.csv")


if __name__ == "__main__":
    main()
